##Self-Query

---

The system turns the user’s question into a structured internal query before searching your data. Instead of just matching words, it tries to understand what the question is really asking

###Step 1: Install
We need LangChain wrappers + Chroma DB + OpenAI-compatible support.

In [ ]:
!pip install --quiet langchain-openai langchain-chroma pydantic

###Step 2: Imports + API CONFIG
* ChatOpenAI → to understand user query

* OpenAIEmbeddings → to convert text to vectors

* Chroma → vector database

* Pydantic → enforce structured output

In [ ]:
import os
from typing import Optional, Literal
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
os.environ["OPENAI_API_KEY"]="YOUR_API_KEY"
os.environ["OPENAI_API_BASE"]="https://apidev.navigatelabsai.com"

###Step 3: Data (Small & Clean)
Each document has:

* Content → semantic meaning

* Metadata → filters (year, genre, rating)

In [ ]:
docs=[
    Document(
        page_content="Scientists clone dinosaurs, chaos follows.",
        metadata={"year": 1993, "rating": 7.7, "genre": "sci-fi"}
    ),
    Document(
        page_content="A dream within a dream heist.",
        metadata={"year": 2010, "rating": 8.2, "genre": "sci-fi"}
    ),
    Document(
        page_content="Toys come alive when humans are away.",
        metadata={"year": 1995, "rating": 8.3, "genre": "animated"}
    ),
    Document(
        page_content="Detectives hunt a serial killer.",
        metadata={"year": 1995, "rating": 8.6, "genre": "crime"}
    ),
]


###Step 4: Embeddings + Vector Store
* Convert text → vectors

* Store them for semantic similarity search

**Why did we use text-embedding-3-small?**

  * It converts text into vectors for similarity search
  * We used text-embedding-3-small because it is an efficient, supported embedding model optimized for semantic search and retrieval tasks, and it is compatible with our API key.

In [ ]:
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://apidev.navigatelabsai.com"
)
vectorstore=Chroma.from_documents(docs, embeddings)

###Step 5: Self-Query Brain (Structured Output)
We force the LLM to extract filters instead of guessing.

In [ ]:
class SearchMetadata(BaseModel):
    query: str=Field(description="What the movie is about")
    genre: Optional[Literal["sci-fi", "animated", "crime"]]=None
    year: Optional[int]=None
    min_rating: Optional[float]=None
llm=ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0,
    base_url="https://apidev.navigatelabsai.com"
)
query_analyzer=llm.with_structured_output(SearchMetadata)


###Step 6: Search Function (Manual Self-Query Retrieval)
This connects:

* User query → LLM

* LLM output → vector DB filters

In [ ]:
def search(user_prompt):
    print("User Query:", user_prompt)
    parsed=query_analyzer.invoke(user_prompt) # Step A: LLM extracts structured filters
    print("Parsed Query:", parsed)
    filters=[]
    if parsed.genre:
        filters.append({"genre": parsed.genre})
    if parsed.year:
        filters.append({"year": parsed.year})
    if parsed.min_rating:
        filters.append({"rating": {"$gte": parsed.min_rating}})
    if len(filters) == 0: # Step B: Build Chroma-compatible where clause
        where_clause=None
    elif len(filters) == 1:
        where_clause=filters[0]
    else:
        where_clause={"$and": filters}
    results=vectorstore.similarity_search(  # Step C: Vector search
        parsed.query,
        k=2,
        filter=where_clause
    )
    return results

###Step 7: Run It
Demonstrates Self-Query Retrieval.

In [ ]:
results=search("I want a sci-fi movie from 1993 about dinosaurs")

print("\n--- RESULTS ---")
for r in results:
    print(r.page_content)
    print("Metadata:", r.metadata)

###Summary:
**What Is Happening Internally ?**

* Take the user’s natural language question

* LLM analyzes the question and extracts intent + metadata filters

* Convert extracted filters into structured query conditions

* Apply metadata filters to the vector database

* Perform semantic similarity search on filtered data

* Return the most relevant documents

* User query → LLM extracts filters → filtered vector search → results